## 1. Загрузка обработанных данных

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

P = Path("data") / "processed"

pairs       = pd.read_parquet(P / "train_pairs.parquet")   # запрос -> item_id
train_items = pd.read_parquet(P / "train_items.parquet")   # объявления из train
items       = pd.read_parquet(P / "items.parquet")         # корпус benchmark
queries     = pd.read_parquet(P / "queries.parquet")       # запросы benchmark

print("pairs:", pairs.shape, "| train_items:", train_items.shape,
      "| items:", items.shape, "| queries:", queries.shape)

pairs: (467054, 10) | train_items: (344825, 18) | items: (189212, 18) | queries: (2452, 10)


## 2. Поисковые сессии


In [2]:
Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]

pairs["q_key"] = pairs.groupby(Q_COLS, dropna=False).ngroup()

relevant = pairs.groupby("q_key")["item_id"].agg(lambda s: sorted(set(s)))
sessions = (pairs.drop_duplicates("q_key")
                 .drop(columns="item_id")
                 .set_index("q_key")
                 .assign(relevant=relevant)
                 .reset_index())

print("Сессий:", len(sessions))
print("Уникальных текстов запросов:", sessions["q_clean"].nunique())
display(sessions[["q_clean", "search_location_id", "search_vid", "relevant"]].head())

Сессий: 354463
Уникальных текстов запросов: 73868


,q_clean,search_location_id,search_vid,relevant
0,скупка телевизоров,652430,,[e8b685dffe1a408e]
1,автоподбор,640860,,[92e1b0446f827b59]
2,баня на дровах,653240,"красота, здоровье","[4c38db95d4d6dd15, 624846856ce81d69, 9fe6eb000d2e1cd7]"
3,изготовление госномера на авто,634670,"оборудование, производство",[45b8628b9c6e7b85]
4,укладка плитки,658430,ремонт и отделка,[c95a4a7daf2a967f]


## 3. Отбираем валидационные запросы

In [3]:
SEED = 42
N_VAL = 5000
TARGET_VID_SHARE = (queries["search_vid"] != "").mean()          # доля с фильтром в benchmark
TARGET_SEEN_SHARE = queries["q_clean"].isin(set(pairs["q_clean"])).mean()  # доля знакомых текстов
print(f"Цели: доля с фильтром по виду услуги = {TARGET_VID_SHARE:.3f}, "
      f"доля знакомых текстов = {TARGET_SEEN_SHARE:.3f}")

rng = np.random.default_rng(SEED)

# 3.1 — одна случайная сессия на каждый уникальный текст
one_per_text = (sessions.assign(_rnd=rng.random(len(sessions)))
                        .sort_values("_rnd")
                        .drop_duplicates("q_clean")
                        .drop(columns="_rnd"))

# 3.2 — нужная доля запросов с фильтром по виду услуги
with_vid = one_per_text[one_per_text["search_vid"] != ""]
without_vid = one_per_text[one_per_text["search_vid"] == ""]
n_vid = int(round(N_VAL * TARGET_VID_SHARE))
val = pd.concat([
    with_vid.sample(min(n_vid, len(with_vid)), random_state=SEED),
    without_vid.sample(min(N_VAL - n_vid, len(without_vid)), random_state=SEED),
]).reset_index(drop=True)

# 3.3 — нужная доля знакомых текстов
sessions_per_text = sessions["q_clean"].value_counts()
val["seen"] = val["q_clean"].map(sessions_per_text) > 1     # есть другие сессии с этим текстом
natural_seen = val["seen"].mean()
print(f"Естественная доля знакомых текстов: {natural_seen:.3f}")

if natural_seen > TARGET_SEEN_SHARE:
    n_make_unseen = int(round((natural_seen - TARGET_SEEN_SHARE) * len(val)))
    make_unseen = val[val["seen"]].sample(n_make_unseen, random_state=SEED).index
    val.loc[make_unseen, "seen"] = False

# Обучающая часть: всё, кроме валидационных сессий и всех сессий «незнакомых» текстов
unseen_texts = set(val.loc[~val["seen"], "q_clean"])
drop = pairs["q_key"].isin(set(val["q_key"])) | pairs["q_clean"].isin(unseen_texts)
train_part = pairs[~drop].reset_index(drop=True)

print(f"\nВалидационных запросов: {len(val)}")
print(f"Обучающая часть: {len(train_part)} строк из {len(pairs)} "
      f"(убрано {drop.mean():.1%})")

Цели: доля с фильтром по виду услуги = 0.340, доля знакомых текстов = 0.374
Естественная доля знакомых текстов: 0.368

Валидационных запросов: 5000
Обучающая часть: 461747 строк из 467054 (убрано 1.1%)


### Проверяем, что валидация похожа на benchmark

Это самая важная ячейка шага. Если цифры в двух колонках сильно расходятся, выводам валидации доверять нельзя.

In [4]:
val_items = set(i for rel in val["relevant"] for i in rel)
train_part_texts = set(train_part["q_clean"])

compare = pd.DataFrame({
    "benchmark": {
        "запросов": len(queries),
        "уникальных текстов, доля": queries["q_clean"].nunique() / len(queries),
        "с фильтром по виду услуги": (queries["search_vid"] != "").mean(),
        "текст встречался в train": queries["q_clean"].isin(set(pairs["q_clean"])).mean(),
        "средняя длина запроса, слов": queries["q_clean"].str.split().str.len().mean(),
        "категория 0": (queries["search_category"] == 0).mean(),
    },
    "валидация": {
        "запросов": len(val),
        "уникальных текстов, доля": val["q_clean"].nunique() / len(val),
        "с фильтром по виду услуги": (val["search_vid"] != "").mean(),
        "текст встречался в train": val["q_clean"].isin(train_part_texts).mean(),
        "средняя длина запроса, слов": val["q_clean"].str.split().str.len().mean(),
        "категория 0": (val["search_category"] == 0).mean(),
    },
}).round(3)
display(compare)

print(f"Правильных ответов на запрос: в среднем {val['relevant'].str.len().mean():.2f}")
print(f"Правильные ответы валидации, встречавшиеся в обучающей части: "
      f"{pd.Series(list(val_items)).isin(set(train_part['item_id'])).mean():.3f} "
      f"(в benchmark: около 0.096 объявлений корпуса встречались в train)")

,benchmark,валидация
запросов,2452.000,5000.000
"уникальных текстов, доля",1.000,1.000
с фильтром по виду услуги,0.340,0.340
текст встречался в train,0.374,0.368
"средняя длина запроса, слов",3.197,3.057
категория 0,0.091,0.000


Правильных ответов на запрос: в среднем 1.06
Правильные ответы валидации, встречавшиеся в обучающей части: 0.387 (в benchmark: около 0.096 объявлений корпуса встречались в train)


## 4. Корпус для валидации

In [5]:
extra = train_items[train_items["item_id"].isin(val_items)
                    & ~train_items["item_id"].isin(set(items["item_id"]))]
val_corpus = pd.concat([items, extra], ignore_index=True)

print(f"Корпус benchmark: {len(items)}, добавлено ответов валидации: {len(extra)}, "
      f"итого: {len(val_corpus)}")

Корпус benchmark: 189212, добавлено ответов валидации: 4880, итого: 194092


## 5. Метрика recall@50

Для каждого запроса смотрим, какая доля его правильных ответов попала в наши 50 кандидатов

In [6]:
def evaluate(predictions: dict, val: pd.DataFrame, k: int = 50, verbose: bool = True):
    """Считает recall@k. predictions: {q_key: список item_id по убыванию уверенности}."""
    hits, n_rel = [], []
    for q_key, rel in zip(val["q_key"], val["relevant"]):
        cand = set(predictions.get(q_key, [])[:k])
        rel = set(rel)
        hits.append(len(cand & rel))
        n_rel.append(len(rel))
    hits, n_rel = np.array(hits), np.array(n_rel)
    per_query = pd.Series(hits / n_rel, index=val.index)

    result = {"recall_macro": per_query.mean(), "recall_micro": hits.sum() / n_rel.sum()}
    if verbose:
        groups = {
            "знакомый текст": val["seen"],
            "новый текст": ~val["seen"],
            "с фильтром вида услуги": val["search_vid"] != "",
            "без фильтра": val["search_vid"] == "",
        }
        print(f"recall@{k}: macro = {result['recall_macro']:.4f}, micro = {result['recall_micro']:.4f}")
        for name, mask in groups.items():
            print(f"   {name:24s} {per_query[mask].mean():.4f}  (запросов: {mask.sum()})")
    return result, per_query

## 6. Sanity check: случайное угадывание

Прежде чем доверять метрике, проверим её на решении, результат которого мы знаем заранее. Если выбирать 50 объявлений случайно из ~190 тысяч, recall должен быть около 50 / 190 000 ≈ 0.0003. Если выйдет что-то сильно больше, в коде ошибка.

In [7]:
all_ids = val_corpus["item_id"].to_numpy()
random_pred = {qk: list(rng.choice(all_ids, 50, replace=False)) for qk in val["q_key"]}
_ = evaluate(random_pred, val)
print(f"Ожидали около {50 / len(val_corpus):.4f}")

recall@50: macro = 0.0002, micro = 0.0002
   знакомый текст           0.0000  (запросов: 1840)
   новый текст              0.0003  (запросов: 3160)
   с фильтром вида услуги   0.0000  (запросов: 1699)
   без фильтра              0.0003  (запросов: 3301)
Ожидали около 0.0003


## 7. Потолки фильтров

In [8]:
# Локация и координаты каждого объявления валидационного корпуса
loc_of = val_corpus.set_index("item_id")["item_location_id"].to_dict()
lat_of = val_corpus.set_index("item_id")["item_latitude"].to_dict()
lon_of = val_corpus.set_index("item_id")["item_longitude"].to_dict()

# Какие локации объявлений выбирали из каждой локации поиска (только обучающая часть!)
tp = train_part[["search_location_id", "item_id"]].merge(
    train_items[["item_id", "item_location_id", "item_latitude", "item_longitude"]], on="item_id")
loc_counts = tp.groupby(["search_location_id", "item_location_id"]).size()
loc_share = loc_counts / loc_counts.groupby(level=0).transform("sum")

def build_loc_map(min_share):
    """search_location_id -> множество разрешённых item_location_id."""
    kept = loc_share[loc_share >= min_share].reset_index()
    m = kept.groupby("search_location_id")["item_location_id"].agg(set).to_dict()
    return m

# «Центр» локации поиска: медиана координат выбранных оттуда объявлений
centers = tp.groupby("search_location_id")[["item_latitude", "item_longitude"]].median()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

print("Локаций поиска в обучающей части:", centers.shape[0])
print("Валидационных запросов, чья локация есть в обучающей части:",
      f"{val['search_location_id'].isin(centers.index).mean():.3f}")

Локаций поиска в обучающей части: 2777
Валидационных запросов, чья локация есть в обучающей части: 0.999


### Размер пула

Пул зависит только от локации поиска, поэтому считаем его один раз на каждую уникальную локацию, а не на каждый запрос. Для поиска по радиусу используем `BallTree` из `scikit-learn`: это структура данных, которая быстро находит все точки в заданном радиусе, не перебирая все 190 тысяч объявлений.

In [9]:
from sklearn.neighbors import BallTree

EARTH_R = 6371.0
corpus_loc = val_corpus["item_location_id"].to_numpy()
# NaN-координаты заменяем на (0, 0): такие объявления просто не попадут ни в один радиус
coords = np.radians(val_corpus[["item_latitude", "item_longitude"]].fillna(0).to_numpy())
tree = BallTree(coords, metric="haversine")


def pool_mask(search_loc, loc_map=None, radius_km=None):
    """Булева маска объявлений корпуса, прошедших фильтр по локации."""
    allowed = {search_loc} | (loc_map.get(search_loc, set()) if loc_map is not None else set())
    mask = np.isin(corpus_loc, list(allowed))
    if radius_km is not None and search_loc in centers.index:
        c = np.radians(centers.loc[search_loc].to_numpy())[None, :]
        idx = tree.query_radius(c, r=radius_km / EARTH_R)[0]
        mask[idx] = True
    return mask


def passes(item_id, search_loc, loc_map=None, radius_km=None):
    """Проходит ли конкретное объявление через фильтр по локации."""
    loc = loc_of.get(item_id)
    if loc == search_loc or (loc_map is not None and loc in loc_map.get(search_loc, set())):
        return True
    if radius_km is not None and search_loc in centers.index:
        c_lat, c_lon = centers.loc[search_loc]
        return haversine_km(c_lat, c_lon, lat_of[item_id], lon_of[item_id]) <= radius_km
    return False


def evaluate_filter(name, loc_map=None, radius_km=None):
    # потолок recall (macro): доля правильных ответов, прошедших фильтр
    ceil = np.mean([
        np.mean([passes(i, s, loc_map, radius_km) for i in rel])
        for s, rel in zip(val["search_location_id"], val["relevant"])
    ])
    # размер пула — по уникальным локациям, потом разворачиваем на запросы
    sizes = {s: pool_mask(s, loc_map, radius_km).sum() for s in val["search_location_id"].unique()}
    pool = val["search_location_id"].map(sizes)
    return {"фильтр": name, "потолок recall": round(ceil, 4),
            "пул: медиана": int(pool.median()), "пул: p90": int(pool.quantile(0.9)),
            "пустой пул": round((pool == 0).mean(), 3)}

In [10]:
%%time
loc_map_1 = build_loc_map(0.01)    # локации, куда уходит >= 1% выборов
loc_map_01 = build_loc_map(0.001)  # >= 0.1% выборов

results = [
    evaluate_filter("без фильтра (весь корпус)", loc_map=None, radius_km=40000),
    evaluate_filter("точная локация"),
    evaluate_filter("соответствие, доля >= 1%", loc_map=loc_map_1),
    evaluate_filter("соответствие, доля >= 0.1%", loc_map=loc_map_01),
    evaluate_filter("радиус 30 км", radius_km=30),
    evaluate_filter("радиус 50 км", radius_km=50),
    evaluate_filter("радиус 100 км", radius_km=100),
    evaluate_filter("соответствие 1% + радиус 50 км", loc_map=loc_map_1, radius_km=50),
    evaluate_filter("соответствие 0.1% + радиус 100 км", loc_map=loc_map_01, radius_km=100),
]
filters_table = pd.DataFrame(results)
display(filters_table)

,фильтр,потолок recall,пул: медиана,пул: p90,пустой пул
0,без фильтра (весь корпус),0.9992,194092,194092,0.000
1,точная локация,0.7656,1126,20289,0.167
2,"соответствие, доля >= 1%",0.9244,3136,22023,0.000
3,"соответствие, доля >= 0.1%",0.9649,26090,41715,0.000
4,радиус 30 км,0.9105,2674,28093,0.000
5,радиус 50 км,0.9313,2859,33724,0.000
6,радиус 100 км,0.9481,3985,40320,0.000
7,соответствие 1% + радиус 50 км,0.9537,3619,33724,0.000
8,соответствие 0.1% + радиус 100 км,0.9784,27076,52875,0.000


CPU times: total: 4 s
Wall time: 4 s


### Кого теряем даже с самым мягким фильтром

Посмотрим на правильные ответы, которые не проходят через мягкий вариант. Если среди них много онлайн-услуг (психолог онлайн, репетитор, дизайнер), для них позже сделаем отдельную ветку поиска без фильтра по локации.

In [11]:
lost_rows = [
    (q, i) for q, s, rel in zip(val["q_clean"], val["search_location_id"], val["relevant"])
    for i in rel if not passes(i, s, loc_map_1, 50)
]
lost = pd.DataFrame(lost_rows, columns=["q_clean", "item_id"]).merge(
    val_corpus[["item_id", "title", "vid"]], on="item_id")
print("Потеряно ответов:", len(lost))
display(lost["vid"].value_counts().head(10))
display(lost.sample(min(20, len(lost)), random_state=SEED))

Потеряно ответов: 244


vid
автосервис, аренда            36
обучение, курсы               26
деловые услуги                24
оборудование, производство    18
строительство                 17
                              15
праздники, мероприятия        15
сад, благоустройство          13
грузоперевозки                11
ремонт и отделка              11
Name: count, dtype: int64

,q_clean,item_id,title,vid
24,пробить скважину,e05eea0946cfc1ce,бурение скважин,"сад, благоустройство"
6,помощь в постановке на учет в гибдд,4ce3ea45dfc70104,услуги по помощи в гибдд,деловые услуги
153,йога тур на алтай,9b8530046d5b2afd,йога на алтае,
211,чертежи студентам nanocad,af926cc00c8cfc78,"чертежи и 3d моделирование (компас, nx)",деловые услуги
198,пошив тентов для беседок,3d989fd5239a8a39,"изготовление тентов, палаток, из пвх","оборудование, производство"
176,реставрация столешниц из камня,2fcdf669db8f1f50,"кристализация камня, полировка плитки",ремонт и отделка
192,металлообработка на станках с чпу,17327001d3fc9628,токарные работы фрезерные работы чпу эрозионная,"оборудование, производство"
124,грузоперевозки 20тн,c03e855537232926,грузоперевозки газель фура попутный груз межгород,грузоперевозки
9,устранение масложора,36e1866666fbfb94,раскоксовка. масложору нет водородом. без разбора,"автосервис, аренда"
101,ремонт ходовой газель,ffbdbb9d8910db98,ремонт газелей и легковых автомобилей,"автосервис, аренда"


### Фильтр по виду услуги

Для запросов, где пользователь выставил вид услуги, проверяем два варианта: строгий и мягкий

In [12]:
vid_of = val_corpus.set_index("item_id")["vid"].to_dict()
corpus_vid = val_corpus["vid"].to_numpy()
vq = val[val["search_vid"] != ""]

for name, allow_empty in [("строгий", False), ("мягкий (+ пустой вид)", True)]:
    ceil = np.mean([
        np.mean([vid_of.get(i) == v or (allow_empty and vid_of.get(i) == "") for i in rel])
        for v, rel in zip(vq["search_vid"], vq["relevant"])
    ])
    share = np.mean([
        ((corpus_vid == v) | (allow_empty & (corpus_vid == ""))).mean() for v in vq["search_vid"]
    ])
    print(f"{name:24s} потолок recall: {ceil:.4f} | остаётся корпуса: {share:.3f}")

строгий                  потолок recall: 0.9800 | остаётся корпуса: 0.069
мягкий (+ пустой вид)    потолок recall: 0.9812 | остаётся корпуса: 0.133


## 8. Сохраняем

In [13]:
val.to_parquet(P / "val_queries.parquet", index=False)
train_part.to_parquet(P / "train_part.parquet", index=False)
val_corpus.to_parquet(P / "val_corpus.parquet", index=False)

for f in ["val_queries.parquet", "train_part.parquet", "val_corpus.parquet"]:
    print(f"{f:22s} {(P / f).stat().st_size / 1e6:8.1f} MB")

val_queries.parquet         0.4 MB
train_part.parquet         27.5 MB
val_corpus.parquet        313.8 MB
